In [3]:
import cv2 as cv
import numpy as np

img = np.zeros((200,200,3), dtype=np.uint8)

# cv.imshow("Image",img)
# cv.waitKey(0)

def update(val):
    value = cv.getTrackbarPos("Threshold","Window")

    display = img.copy()
    # cv.imshow("Image",img)
    # cv.waitKey(0)
    
    cv.putText(display, f"Value:{value}",(50,100),cv.FONT_HERSHEY_SIMPLEX, 1, (0,255,45), 2)
    cv.imshow("Window",display)
cv.namedWindow("Window") # Hey pc, loof for a window named "Window"
cv.createTrackbar("Threshold","Window", 0, 255, update)

update(0)
cv.waitKey(0)
cv.destroyAllWindows()



In [11]:
import cv2 as cv
import numpy as np

image = cv.imread("bloodcell2.jpg")

# cv.imshow("Image",img)
# cv.waitKey(0)

gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY) 

# cv.imshow("Image",img)
# cv.waitKey(0)

def update(val):
    thresh_value = cv.getTrackbarPos("Threshold","Segmentation")

    if thresh_value == 0:
        text = "Otsu"
        _, binary = cv.threshold(gray, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    else:
        text = "Manual"
        _, binary = cv.threshold(gray, thresh_value, 255, cv.THRESH_BINARY)

    binary_inv = cv.bitwise_not(binary)
    contours, _ = cv.findContours(binary_inv, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    
    output = image.copy()
    object_lengths = []

    for cnt in contours:
        if cv.contourArea(cnt) < 15: # adjust if you wish
            continue
        rect = cv.minAreaRect(cnt)     # bounding box
        (x, y), (w, h), angle = rect

        # find out the length of the object
        max_len = max(w, h)
        object_lengths.append(max_len)

        # Draw rectangle + size
        box = cv.boxPoints(rect)
        box = np.intp(box)
        cv.drawContours(output, [box], 0, (0,255,0), 2)
        cv.putText(output, f"{max_len:.1f}", (int(x), int(y)),
                   cv.FONT_HERSHEY_SIMPLEX, 0.4, (0,0,255), 2)

    # Show info
    cv.putText(output, f"{text} | Objects: {len(object_lengths)}",
               (10, 25), cv.FONT_HERSHEY_SIMPLEX, 0.8, (255,0,0), 2)

    cv.imshow("Segmentation", output)
    cv.imshow("Binary", binary_inv)

cv.namedWindow("Segmentation", cv.WINDOW_NORMAL)
cv.namedWindow("Binary", cv.WINDOW_NORMAL)

cv.createTrackbar("Threshold", "Segmentation", 0, 255, update)
update(0)   # initial run
cv.waitKey(0)
cv.destroyAllWindows()

